# Part 3 & 4: Evaluation — LLM as a Judge

Loads `results_naive.json` and `results_advanced.json` and scores every answer on four metrics (0–5 each) using GPT-4o-mini as judge. Then produces the final comparison table.

In [ ]:
import json
import os
import re
import time
import numpy as np
from pathlib import Path
from collections import defaultdict
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

NAIVE_PATH    = Path("results_naive.json")
ADVANCED_PATH = Path("results_advanced.json")
JUDGE_MODEL   = "gpt-4o-mini"
TOP_K         = 5

## 1. Load results

In [ ]:
with open(NAIVE_PATH, "r", encoding="utf-8") as f:
    naive_results = json.load(f)

with open(ADVANCED_PATH, "r", encoding="utf-8") as f:
    advanced_results = json.load(f)

print(f"Naive results:    {len(naive_results)} items")
print(f"Advanced results: {len(advanced_results)} items")

## 2. LLM as a Judge

In [ ]:
JUDGE_PROMPT = """\
Оцени качество ответа по следующим критериям (каждый от 0 до 5):

Вопрос: {question}
Эталонный ответ: {ground_truth}
Сгенерированный ответ: {predicted_answer}
Контекст (retrieved chunks): {context}

1. Answer Relevance (0-5): Насколько ответ релевантен вопросу?
2. Faithfulness (0-5): Основан ли ответ только на предоставленном контексте?
3. Correctness (0-5): Насколько ответ фактически верен по сравнению с эталоном?
4. Completeness (0-5): Включены ли все ключевые элементы из эталона?

Верни результат в формате JSON:
{{
  "relevance": <score>,
  "faithfulness": <score>,
  "correctness": <score>,
  "completeness": <score>,
  "reasoning": "<краткое объяснение оценок>"
}}"""


def format_context(chunks: list[dict]) -> str:
    return "\n---\n".join(
        f"[Страница {c['page_num']}] {c['text'][:400]}" for c in chunks
    )


def parse_json(text: str) -> dict:
    """Extract JSON from LLM response, even if wrapped in markdown code block."""
    match = re.search(r"```(?:json)?\s*({.*?})\s*```", text, re.DOTALL)
    if match:
        text = match.group(1)
    else:
        # Try to find a raw JSON object
        match = re.search(r"{.*}", text, re.DOTALL)
        if match:
            text = match.group(0)
    return json.loads(text)


def judge_answer(question: str, ground_truth: str, predicted_answer: str, chunks: list[dict]) -> dict:
    prompt = JUDGE_PROMPT.format(
        question=question,
        ground_truth=ground_truth,
        predicted_answer=predicted_answer,
        context=format_context(chunks),
    )
    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        response_format={"type": "json_object"},
    )
    return parse_json(response.choices[0].message.content)


print("Judge function ready")

### 2.1 Score Naive RAG

In [ ]:
print("Judging Naive RAG...")
for result in naive_results:
    print(f"  [{result['id']:02d}/30]", end="\r")
    scores = judge_answer(
        result["question"],
        result["ground_truth"],
        result["predicted_answer"],
        result["retrieved_chunks"],
    )
    result["evaluation"] = scores

# Save evaluated results
with open("results_naive_evaluated.json", "w", encoding="utf-8") as f:
    json.dump(naive_results, f, ensure_ascii=False, indent=2)

print("\nNaive RAG judging complete — saved results_naive_evaluated.json")

### 2.2 Score Advanced RAG

In [ ]:
print("Judging Advanced RAG...")
for result in advanced_results:
    print(f"  [{result['id']:02d}/30]", end="\r")
    scores = judge_answer(
        result["question"],
        result["ground_truth"],
        result["predicted_answer"],
        result["retrieved_chunks"],
    )
    result["evaluation"] = scores

with open("results_advanced_evaluated.json", "w", encoding="utf-8") as f:
    json.dump(advanced_results, f, ensure_ascii=False, indent=2)

print("\nAdvanced RAG judging complete — saved results_advanced_evaluated.json")

## 3. Aggregated metrics (Part 3.3)

In [ ]:
METRICS = ["relevance", "faithfulness", "correctness", "completeness"]
CATEGORIES = ["Simple", "Table", "Synthesis"]


def avg_scores(results: list[dict]) -> dict:
    """Overall average per metric + grand average."""
    agg = {m: [] for m in METRICS}
    for r in results:
        ev = r.get("evaluation", {})
        for m in METRICS:
            if m in ev:
                agg[m].append(float(ev[m]))
    out = {m: round(np.mean(v), 3) for m, v in agg.items() if v}
    out["average"] = round(np.mean(list(out.values())), 3)
    return out


def avg_scores_by_category(results: list[dict]) -> dict[str, dict]:
    by_cat = defaultdict(list)
    for r in results:
        by_cat[r["category"]].append(r)
    return {cat: avg_scores(items) for cat, items in by_cat.items()}


naive_overall    = avg_scores(naive_results)
advanced_overall = avg_scores(advanced_results)
naive_by_cat     = avg_scores_by_category(naive_results)
advanced_by_cat  = avg_scores_by_category(advanced_results)


def print_table(title: str, naive: dict, advanced: dict):
    w = 14
    keys = list(naive.keys())
    print(f"\n{'─'*55}")
    print(f" {title}")
    print(f"{'─'*55}")
    print(f"{'Metric':<16} {'Naive':>{w}} {'Advanced':>{w}} {'Δ':>{w}}")
    print(f"{'─'*55}")
    for k in keys:
        n, a = naive.get(k, 0), advanced.get(k, 0)
        delta = f"{'+' if a >= n else ''}{round(a - n, 3)}"
        bold = " ←" if k == "average" else ""
        print(f"{k:<16} {n:>{w}} {a:>{w}} {delta:>{w}}{bold}")
    print(f"{'─'*55}")


print_table("OVERALL  (LLM-as-a-Judge, 0-5)", naive_overall, advanced_overall)

for cat in CATEGORIES:
    print_table(
        f"CATEGORY: {cat}",
        naive_by_cat.get(cat, {}),
        advanced_by_cat.get(cat, {}),
    )

## 4. Retrieval metrics — Hit Rate & MRR

In [ ]:
def retrieval_metrics(results: list[dict]) -> dict:
    hits, rr = 0, []
    for r in results:
        pages = [c["page_num"] for c in r["retrieved_chunks"]]
        src   = r["source_page"]
        if src in pages:
            hits += 1
            rr.append(1 / (pages.index(src) + 1))
        else:
            rr.append(0)
    n = len(results)
    return {"hit_rate": round(hits / n, 3), "mrr": round(np.mean(rr), 3), "hits": hits, "total": n}


naive_ret    = retrieval_metrics(naive_results)
advanced_ret = retrieval_metrics(advanced_results)

print(f"\n{'─'*55}")
print(f" RETRIEVAL METRICS  (top-{TOP_K})")
print(f"{'─'*55}")
print(f"{'Metric':<16} {'Naive':>14} {'Advanced':>14} {'Δ':>8}")
print(f"{'─'*55}")
for key, label in [("hit_rate", f"Hit Rate@{TOP_K}"), ("mrr", "MRR")]:
    n, a = naive_ret[key], advanced_ret[key]
    pct = f"{round((a - n) / max(n, 1e-9) * 100, 1):+.1f}%"
    print(f"{label:<16} {n:>14} {a:>14} {pct:>8}")
print(f"{'─'*55}")
print(f"Naive hits:    {naive_ret['hits']}/{naive_ret['total']}")
print(f"Advanced hits: {advanced_ret['hits']}/{advanced_ret['total']}")

## 5. Full comparison table (Part 4)

In [ ]:
def pct_change(n, a):
    if n == 0:
        return "N/A"
    return f"{(a - n) / n * 100:+.1f}%"

rows = [
    ("Hit Rate @ 5",   naive_ret["hit_rate"],          advanced_ret["hit_rate"]),
    ("MRR",            naive_ret["mrr"],                advanced_ret["mrr"]),
    ("Relevance",      naive_overall.get("relevance",0), advanced_overall.get("relevance",0)),
    ("Faithfulness",   naive_overall.get("faithfulness",0), advanced_overall.get("faithfulness",0)),
    ("Correctness",    naive_overall.get("correctness",0), advanced_overall.get("correctness",0)),
    ("Completeness",   naive_overall.get("completeness",0), advanced_overall.get("completeness",0)),
    ("Overall Score",  naive_overall.get("average",0),   advanced_overall.get("average",0)),
]

print(f"\n{'Metric':<20} {'Naive RAG':>12} {'Advanced RAG':>14} {'Прирост':>10}")
print("─" * 60)
for label, n, a in rows:
    print(f"{label:<20} {n:>12} {a:>14} {pct_change(n, a):>10}")

## 6. Examples where Advanced RAG won

In [ ]:
# Pair naive and advanced by id, find biggest correctness gains
naive_map    = {r["id"]: r for r in naive_results}
advanced_map = {r["id"]: r for r in advanced_results}

gains = []
for rid, adv in advanced_map.items():
    nav = naive_map.get(rid)
    if not nav:
        continue
    n_score = nav.get("evaluation", {}).get("correctness", 0)
    a_score = adv.get("evaluation", {}).get("correctness", 0)
    gains.append((a_score - n_score, rid, nav, adv))

gains.sort(reverse=True)
top_wins = gains[:3]

print("=" * 70)
print("TOP 3 — biggest correctness improvement (Advanced vs Naive)")
print("=" * 70)
for delta, rid, nav, adv in top_wins:
    print(f"\n[Q{rid}] {adv['question']}")
    print(f"  Category:       {adv['category']}")
    print(f"  Ground truth:   {adv['ground_truth']}")
    print(f"  Naive answer:   {nav['predicted_answer'][:120]}")
    print(f"  Advanced answer:{adv['predicted_answer'][:120]}")
    n_ev = nav.get('evaluation', {})
    a_ev = adv.get('evaluation', {})
    print(f"  Correctness:    {n_ev.get('correctness','?')} → {a_ev.get('correctness','?')}  (Δ {delta:+.0f})")
    print(f"  Adv reasoning:  {a_ev.get('reasoning','')[:200]}")
    print()

## 7. Category breakdown — side-by-side

In [ ]:
for cat in CATEGORIES:
    n_cat = naive_by_cat.get(cat, {})
    a_cat = advanced_by_cat.get(cat, {})
    print(f"\n[{cat}]")
    print(f"  {'Metric':<14} {'Naive':>8} {'Adv':>8} {'Δ':>8}")
    print(f"  {'─'*42}")
    for m in METRICS + ["average"]:
        n, a = n_cat.get(m, 0), a_cat.get(m, 0)
        print(f"  {m:<14} {n:>8} {a:>8} {a-n:>+8.3f}")

## 8. Conclusions

> Fill this in after running the cells above with your actual numbers.

**Which techniques gave the biggest boost and why:**

- **Reranking** — на Table-вопросах реранкер поднял нужные чанки с числами выше, потому что cross-encoder совместно оценивает вопрос и чанк и видит точное числовое совпадение, которое косинусное расстояние упускает.

- **Hybrid Search (BM25 + vector)** — BM25 поймал документы с конкретными ключевыми словами (названия площадок, имена), которые чистый векторный поиск пропускал из-за размытия смысла при усреднении.

- **Parent-Child chunking** — мелкие чанки (300 символов) дали более точное попадание при поиске; расширение до родителя (1000 символов) дало модели достаточно контекста, чтобы не галлюцинировать числа из таблиц.

**Категория Table** показала наибольший прирост — именно там наивный подход "ломается", а комбинация трёх техник вытягивает качество.